In [ ]:
#Naive Bayes
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans # Uniquement pour générer des labels de test

#chrger la dataset
df = pd.read_csv('wine-clustering.csv')

#On sélectionne deux variables pour la visualisation 2D
features = ['Alcohol', 'Color_Intensity']
X = df[features].values

#Puisque c'est du clustering, on crée 3 classes cibles pour l'entraînement
kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
y = kmeans.labels_

#Implémentation de Naive Bayes
class GaussianNaiveBayes:

    #fonction d'entraînement
    def fit(self, X, y):

        # récupérer toutes les classes uniques
        self.classes = np.unique(y)
        # liste pour stocker les informations de chaque classe
        self.stats = []

        for c in self.classes:

            # récupérer seulement les lignes de la classe c
            X_c = X[y == c]

            class_stats = {

                # moyenne de chaque colonne
                'mean': X_c.mean(axis=0),

                # variance de chaque colonne
                'var': X_c.var(axis=0),

                # probabilité de la classe =  nombre éléments classe / nombre total
                'prior': X_c.shape[0] / X.shape[0]
            }

            # ajouter les statistiques dans la liste
            self.stats.append(class_stats)

    # fonction de la distribution gaussienne
    def _gaussian_pdf(self, x, mean, var):

        # petite valeur pour éviter division par 0
        eps = 1e-6

        # première partie de la formule gaussienne
        coeff = 1.0 / np.sqrt(2.0 * np.pi * var + eps)

        # deuxième partie de la formule
        exponent = np.exp(-((x - mean)**2) / (2 * var + eps))

        # résultat final de la formule
        return coeff * exponent

    # prédire plusieurs lignes
    def predict(self, X):

        # appliquer _predict_single sur chaque ligne
        y_pred = [self._predict_single(x) for x in X]

        # transformer en numpy array
        return np.array(y_pred)

    # prédiction d'une seule ligne
    def _predict_single(self, x):

        # liste pour stocker les probabilités
        posteriors = []
        # parcourir chaque classe
        for stats in self.stats:
            # calcul du log de la probabilité a priori
            prior = np.log(stats['prior'])
            # calcul de la somme des probabilités gaussiennes
            likelihood = np.sum(
                np.log(
                    self._gaussian_pdf(
                        x,
                        stats['mean'],
                        stats['var']
                    )
                )
            )

            # calcul probabilité finale
            posterior = prior + likelihood

            # ajouter le résultat
            posteriors.append(posterior)

        # retourner la classe avec la plus grande probabilité
        return self.classes[np.argmax(posteriors)]
#Entraînement du modele
model = GaussianNaiveBayes()
model.fit(X, y)

#Visualisation
def plot_boundaries(X, y, model):
    h = .02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(10, 6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='winter')
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='winter', s=50)
    
    plt.xlabel(features[0])
    plt.ylabel(features[1])
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

plot_boundaries(X, y, model)